# Limpieza, análisis y estructuración de datos con Pandas (Instacart)
## Lección 2: Limpieza y estandarización (Laboratorio guiado)

In [1]:
import pandas as pd
from pathlib import Path

# Sistema robusto para encontrar la ruta de los datos automáticamente
if Path('../data/raw').exists():
    RUTA = Path('../data/raw')
    SALIDA = Path('../data/processed')
elif Path('data/raw').exists():
    RUTA = Path('data/raw')
    SALIDA = Path('data/processed')
else:
    raices = (Path.cwd(), *Path.cwd().parents)
    raiz = next((r for r in raices if (r / 'data' / 'raw').is_dir()), Path.cwd())
    RUTA = raiz / 'data' / 'raw'
    SALIDA = raiz / 'data' / 'processed'

SALIDA.mkdir(parents=True, exist_ok=True)

# 1. Cargar datos crudos
orders_crudo = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo = pd.read_csv(RUTA / 'order_products.csv', sep=';')

# 2. Función de registro
registro = []
def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla, 'paso': paso, 'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 2)
    })
    return df

# 3. Funciones de limpieza
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8', 'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True).str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

# 4. Ejecutar Pipeline
orders_limpio = (orders_crudo.pipe(anotar, 'orders', '0 crudo')
                 .pipe(quitar_duplicados).pipe(anotar, 'orders', '1 sin duplicados')
                 .pipe(tipos_orders).pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo.pipe(anotar, 'products', '0 crudo')
                   .pipe(limpiar_nombres).pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo.pipe(anotar, 'order_prods', '0 crudo')
             .pipe(tipos_order_prods).pipe(anotar, 'order_prods', '1 tipos'))

# 5. Mostrar informe
display(pd.DataFrame(registro))

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,22.99
1,orders,1 sin duplicados,478952,28817,22.99
2,orders,2 tipos,478952,28817,7.18
3,products,0 crudo,49694,1258,5.11
4,products,1 nombres,49694,0,5.14
5,order_prods,0 crudo,4545007,836,145.44
6,order_prods,1 tipos,4545007,836,54.54


In [2]:
# Reto 2: Resolviendo las 7 preguntas de la guía
print("--- 1. Duplicados en order_prods ---")
print("Exactos:", op_limpio.duplicated().sum())
print("Pares repetidos:", op_limpio.duplicated(subset=['order_id', 'product_id']).sum())
print("Valores en reordered:", op_limpio['reordered'].unique())

print("\n--- 2. Pasillos y Departamentos ---")
aisles = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')
print("Duplicados aisles:", aisles['aisle'].duplicated().sum(), "| Espacios:", aisles['aisle'].str.contains(r'^\s|\s$', regex=True).sum())
print("Duplicados departments:", departments['department'].duplicated().sum(), "| Espacios:", departments['department'].str.contains(r'^\s|\s$', regex=True).sum())

print("\n--- 3. Nulos en add_to_cart_order ---")
nulos_cart = op_limpio[op_limpio['add_to_cart_order'].isna()]
tamano_pedidos = op_limpio[op_limpio['order_id'].isin(nulos_cart['order_id'])].groupby('order_id').size()
print("Pedidos afectados:", nulos_cart['order_id'].nunique(), "| Mínimo de productos en estos pedidos:", tamano_pedidos.min())
print("Justificación: Son pedidos gigantes de más de 15 artículos, probablemente un error de registro en carritos muy grandes.")

print("\n--- 4. Nombres cambiados al limpiar ---")
diferentes = (products_crudo['product_name'].notna()) & (products_crudo['product_name'] != products_limpio['product_name'])
print("Nombres estandarizados/cambiados:", diferentes.sum())

print("\n--- 5. Memoria: Texto vs Categoría ---")
mapa = {0:'Dom', 1:'Lun', 2:'Mar', 3:'Mie', 4:'Jue', 5:'Vie', 6:'Sab'}
serie_texto = orders_limpio['order_dow'].map(mapa)
serie_cat = pd.Categorical(serie_texto, categories=list(mapa.values()), ordered=True)
print(f"Memoria Texto: {serie_texto.memory_usage(deep=True)/1e6:.1f} MB | Memoria Categoría: {serie_cat.memory_usage(deep=True)/1e6:.1f} MB")

print("\n--- 6. Productos Unknown en pedidos ---")
unk_ids = products_limpio[products_limpio['product_name'] == 'Unknown']['product_id']
unk_pedidos = op_limpio[op_limpio['product_id'].isin(unk_ids)]
print(f"Aparecen en {len(unk_pedidos)} registros. No se deben borrar porque alteraría el total de artículos y facturación del cliente.")

print("\n--- 7. Reemplazo de missing por pd.NA ---")
dept_prueba = departments.copy()
dept_prueba['department'] = dept_prueba['department'].replace('missing', pd.NA)
print("No conviene usar pd.NA en la original antes de agrupar, porque el groupby ignora los nulos por defecto y perderíamos esa data.")

--- 1. Duplicados en order_prods ---
Exactos: 0
Pares repetidos: 0
Valores en reordered: [0 1]

--- 2. Pasillos y Departamentos ---
Duplicados aisles: 0 | Espacios: 0
Duplicados departments: 0 | Espacios: 0

--- 3. Nulos en add_to_cart_order ---
Pedidos afectados: 70 | Mínimo de productos en estos pedidos: 65
Justificación: Son pedidos gigantes de más de 15 artículos, probablemente un error de registro en carritos muy grandes.

--- 4. Nombres cambiados al limpiar ---
Nombres estandarizados/cambiados: 6321

--- 5. Memoria: Texto vs Categoría ---
Memoria Texto: 24.9 MB | Memoria Categoría: 0.5 MB

--- 6. Productos Unknown en pedidos ---
Aparecen en 9547 registros. No se deben borrar porque alteraría el total de artículos y facturación del cliente.

--- 7. Reemplazo de missing por pd.NA ---
No conviene usar pd.NA en la original antes de agrupar, porque el groupby ignora los nulos por defecto y perderíamos esa data.


In [3]:
import numpy as np

# --- Laboratorio 3: Perfil de hábitos de compra ---
# 1. Pedidos por cliente y segmentación
pedidos_por_cliente = orders_limpio.groupby('user_id')['order_id'].count()
segmento_cliente = pd.cut(pedidos_por_cliente, 
                          bins=[0, 1, 4, np.inf], 
                          labels=['ocasional', 'regular', 'frecuente'])

# Llevamos el segmento a cada pedido
orders_limpio['segmento'] = orders_limpio['user_id'].map(segmento_cliente)

# 2. Resumen por segmento
resumen_segmento = (orders_limpio
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
print("--- Resumen por Segmento ---")
display(resumen_segmento)

# 3. Franjas horarias y tabla cruzada
orders_limpio['franja'] = pd.cut(orders_limpio['order_hour_of_day'], 
                                 bins=[-1, 5, 11, 17, 23], 
                                 labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders_limpio['segmento'], orders_limpio['franja'], normalize='index')
          .mul(100).round(1))
print("\n--- Perfil Horario por Segmento (%) ---")
display(perfil)

# 4. Limpieza de auxiliares para no arrastrarlas a la Lección 4
orders_limpio = orders_limpio.drop(columns=['segmento', 'franja'])

--- Resumen por Segmento ---


,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0



--- Perfil Horario por Segmento (%) ---


franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


In [4]:
# --- Soluciones Reto 3 ---
print("--- 1. Horas con más pedidos en domingo ---")
print(orders_limpio[orders_limpio['order_dow'] == 0]['order_hour_of_day'].value_counts().head(3))

print("\n--- 2. Espera media entre pedidos por hora ---")
espera_hora = orders_limpio.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print(f"Mayor espera a las {espera_hora.idxmax()}h | Menor espera a las {espera_hora.idxmin()}h")

print("\n--- 3. Clientes con 10 o más pedidos ---")
print((orders_limpio.groupby('user_id')['order_id'].count() >= 10).sum(), "clientes.")

print("\n--- 4. Pedidos exactamente 7 días después ---")
exactos_7 = (orders_limpio['days_since_prior_order'] == 7).sum()
print(f"{exactos_7} pedidos ({(exactos_7 / len(orders_limpio) * 100):.1f}% del total).")

print("\n--- 5. Pedidos de 20 productos o más ---")
tamano_pedidos = op_limpio.groupby('order_id').size()
print(f"Hay {(tamano_pedidos >= 20).sum()} pedidos masivos. El más grande tiene {tamano_pedidos.max()} productos.")

print("\n--- 6. Tasa de recompra para posiciones 1 a 5 ---")
print(op_limpio[op_limpio['add_to_cart_order'].between(1, 5)].groupby('add_to_cart_order')['reordered'].mean())
print("Patrón: Entre más tarde se agrega al carrito, menor es la probabilidad de recompra.")

print("\n--- 7. Mediana de espera Fines de Semana vs Semana ---")
es_fds = orders_limpio['order_dow'].isin([0, 6])
print("Mediana Fin de Semana:", orders_limpio[es_fds]['days_since_prior_order'].median(), "días.")
print("Mediana Entre Semana:", orders_limpio[~es_fds]['days_since_prior_order'].median(), "días.")

--- 1. Horas con más pedidos en domingo ---
order_hour_of_day
15    7585
14    7533
13    7483
Name: count, dtype: int64

--- 2. Espera media entre pedidos por hora ---
Mayor espera a las 0h | Menor espera a las 22h

--- 3. Clientes con 10 o más pedidos ---
6327 clientes.

--- 4. Pedidos exactamente 7 días después ---
44577 pedidos (9.3% del total).

--- 5. Pedidos de 20 productos o más ---
Hay 49420 pedidos masivos. El más grande tiene 127 productos.

--- 6. Tasa de recompra para posiciones 1 a 5 ---
add_to_cart_order
1    0.678620
2    0.677059
3    0.658986
4    0.637511
5    0.617483
Name: reordered, dtype: float64
Patrón: Entre más tarde se agrega al carrito, menor es la probabilidad de recompra.

--- 7. Mediana de espera Fines de Semana vs Semana ---
Mediana Fin de Semana: 8.0 días.
Mediana Entre Semana: 7.0 días.


In [5]:
# --- Laboratorio 4: Tabla Maestra y Ficha de Productos ---
filas_inicio = len(op_limpio)

# 1. Construir la maestra validada
maestra = (op_limpio
           .merge(products_limpio, on='product_id', how='left', validate='m:1')
           .merge(aisles, on='aisle_id', how='left', validate='m:1')
           .merge(departments, on='department_id', how='left', validate='m:1')
           .merge(orders_limpio, on='order_id', how='left', validate='m:1'))

assert len(maestra) == filas_inicio, "Error: Se multiplicaron o perdieron filas en el cruce."

# 2. Agregar métricas por producto
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())

# 3. Recuperar descriptivos del producto
ficha_productos = (por_producto
    .merge(products_limpio, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes', 'tasa_recompra', 'posicion_media']])

# 4. Encontrar líderes por departamento (>1000 pedidos)
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
populares['puesto'] = populares.groupby('department')['tasa_recompra'].rank(ascending=False, method='first')
lideres = populares[populares['puesto'] == 1].sort_values('tasa_recompra', ascending=False)

print("--- Top 5 Líderes por Departamento ---")
display(lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5))

# 5. Guardar archivos exportados (Con Plan B si falla Parquet)
try:
    maestra.to_parquet(SALIDA / 'instacart_maestra.parquet', index=False)
    print("\nMaestra guardada con éxito en formato Parquet.")
except:
    maestra.to_csv(SALIDA / 'instacart_maestra.csv', index=False, sep=';')
    print("\nLibrería Parquet no detectada. Maestra guardada con éxito en formato CSV.")
lideres.to_csv(SALIDA / 'lideres_por_departamento.csv', index=False, sep=';')

--- Top 5 Líderes por Departamento ---


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004



Librería Parquet no detectada. Maestra guardada con éxito en formato CSV.


In [7]:
# --- Soluciones Reto 4 ---
print("--- 1. Pasillos (>10k artículos): Top 3 y el más bajo en recompra ---")
pasillos = maestra.groupby('aisle').agg(articulos=('order_id', 'size'), tasa_recompra=('reordered', 'mean'))
pasillos_10k = pasillos[pasillos['articulos'] >= 10000].sort_values('tasa_recompra', ascending=False)
print("Top 3:\n", pasillos_10k.head(3))
print("\nEl más bajo:", pasillos_10k.index[-1])

print("\n--- 2. Participación de madrugada (0-5h) vs Total ---")
madrugada = maestra[maestra['order_hour_of_day'].between(0, 5)]
share_madrugada = madrugada['department'].value_counts(normalize=True) * 100
print("Top 3 en madrugada (%):\n", share_madrugada.head(3))

print("\n--- 3. Top 3 productos de snacks ---")
print(maestra[maestra['department'] == 'snacks']['product_name'].value_counts().head(3))

print("\n--- 4. Ficha de clientes ---")
clientes_tb = maestra.groupby('user_id').agg(
    pedidos_distintos=('order_id', 'nunique'),
    articulos=('product_id', 'size'),
    tasa_recompra=('reordered', 'mean')
)
print(f"Total clientes: {len(clientes_tb)} | Promedio artículos: {clientes_tb['articulos'].mean():.1f} | Máx artículos: {clientes_tb['articulos'].max()}")

print("\n--- 5. Inner Join ---")
filas_inner = len(orders_limpio.merge(op_limpio, on='order_id', how='inner'))
print(f"Filas Inner: {filas_inner:,}. Son menos porque el inner descarta cabeceras sin detalle de productos.")

print("\n--- 6. Concat Top 3 Produce y Dairy Eggs ---")
top_produce = maestra[maestra['department'] == 'produce']['product_name'].value_counts().head(3).reset_index().assign(department='produce')
top_dairy = maestra[maestra['department'] == 'dairy eggs']['product_name'].value_counts().head(3).reset_index().assign(department='dairy eggs')
display(pd.concat([top_produce, top_dairy], ignore_index=True))

print("\n--- 7. Tabla Dinámica ---")
pivot = maestra.pivot_table(index='department', columns='order_dow', values='order_id', aggfunc='count')
print(f"Forma de la tabla: {pivot.shape}")
print(f"Día con más pedidos de alcohol: {pivot.loc['alcohol'].idxmax()}")

--- 1. Pasillos (>10k artículos): Top 3 y el más bajo en recompra ---
Top 3:
                                articulos  tasa_recompra
aisle                                                  
milk                              125522       0.781305
water seltzer sparkling water     117133       0.730605
fresh fruits                      510101       0.718197

El más bajo: spices seasonings

--- 2. Participación de madrugada (0-5h) vs Total ---
Top 3 en madrugada (%):
 department
produce       30.205257
dairy eggs    15.873245
beverages      8.489977
Name: proportion, dtype: float64

--- 3. Top 3 productos de snacks ---
product_name
Lightly Salted Baked Snap Pea Crisps    3462
Original Veggie Straws                  2331
Sea Salt Pita Chips                     2275
Name: count, dtype: int64

--- 4. Ficha de clientes ---
Total clientes: 149626 | Promedio artículos: 30.4 | Máx artículos: 580

--- 5. Inner Join ---
Filas Inner: 4,545,007. Son menos porque el inner descarta cabeceras sin detal

,product_name,count,department
0,Banana,66050,produce
1,Bag Of Organic Bananas,53297,produce
2,Organic Strawberries,37039,produce
3,Organic Whole Milk,20953,dairy eggs
4,Organic Half & Half,10752,dairy eggs
5,Half & Half,9588,dairy eggs



--- 7. Tabla Dinámica ---
Forma de la tabla: (21, 7)
Día con más pedidos de alcohol: 5
